# Estágio 03 — Afluências

A estação troca-se na primeira célula de código (`ESTACAO = "ameal"` ou `"antanhol"`). Os CSVs vêm dos estágios 01 e 02.

**Entrada (conforme a estação):**

- `data/interim/<estação>_merged.csv` (estágio 01)
- `data/interim/<estação>_precipitacao.csv` (estágio 02)

O pluviômetro local (`precip_incremental`) passa a ser a série Open-Meteo do estágio 02. A célula do gráfico ainda volta a pedir a Open-Meteo, como no original.


In [1]:
import pandas as pd
from pipeline_config import FIGURES_DIR, INTERIM_DIR, STATIONS

# Alterna a análise: "ameal" ou "antanhol"
ESTACAO = "ameal"

if ESTACAO not in STATIONS:
    raise ValueError(f"Estação desconhecida: {ESTACAO!r}. Opções: {list(STATIONS)}")

STATION = STATIONS[ESTACAO]
STATION_LABEL = STATION["folder"]
CAUDAL_CSV = INTERIM_DIR / f"{ESTACAO}_merged.csv"
PRECIP_CSV = INTERIM_DIR / f"{ESTACAO}_precipitacao.csv"
LATITUDE = STATION["latitude"]
LONGITUDE = STATION["longitude"]

print(f"Estação: {STATION_LABEL} ({ESTACAO})")
print(f"Caudal:  {CAUDAL_CSV}")
print(f"Precip:  {PRECIP_CSV}")
print(f"Coords:  {LATITUDE}, {LONGITUDE}")


Estação: Ameal (ameal)
Caudal:  /Users/marcosgrandesso/wada/afluencia_indevida_pipeline/data/interim/ameal_merged.csv
Precip:  /Users/marcosgrandesso/wada/afluencia_indevida_pipeline/data/interim/ameal_precipitacao.csv
Coords:  40.185668, -8.547218


In [2]:
# Input do estágio 01. O original lia os dois CSVs da curva de vazão
# (altura <= 0.185 e > 0.185); o merge já vem concatenado.
df_caudal = pd.read_csv(CAUDAL_CSV, parse_dates=["datetime"])
df_caudal["data"] = df_caudal["datetime"].dt.strftime("%Y-%m-%d")
df_caudal["hora"] = df_caudal["datetime"].dt.strftime("%H:%M:%S")
df_caudal["caudal_m3_s"] = df_caudal["caudal_l_s"] / 1000.0

# Filtrar a partir de julho/2026: antes disso os sensores tiveram problemas
# (longos buracos entre dez/2025 e jun/2026). Sem limite superior.
ANALISE_INICIO = '2026-07-01'

df_caudal = df_caudal[df_caudal['data'] >= ANALISE_INICIO]

# Descartar caudais <= 0: o medidor nao esta a registar escoamento
# (abaixo do limiar), nao e caudal real.
dias_antes_filtro = set(df_caudal['data'].unique())
n_antes_filtro = len(df_caudal)
df_caudal = df_caudal[df_caudal['caudal_l_s'] > 0]
dias_sem_caudal = sorted(dias_antes_filtro - set(df_caudal['data'].unique()))

df_caudal = df_caudal.sort_values(by=['data', 'hora']).reset_index(drop=True)

print(f"Período em análise: {df_caudal['data'].min()} até {df_caudal['data'].max()}")
print(f"Registos de caudal: {len(df_caudal):,} (descartados {n_antes_filtro - len(df_caudal):,} com caudal <= 0)")
print(f"Dias que ficaram sem nenhuma leitura ({len(dias_sem_caudal)}): "
      + ", ".join(dias_sem_caudal))

# 'data' passa a ser o timestamp completo e 'dia' a data civil: o resto
# do notebook (resample, filtro 02:00-04:00, isin(dias)) conta com as duas.
df_caudal['data'] = pd.to_datetime(df_caudal['data'] + ' ' + df_caudal['hora'])
df_caudal['dia'] = df_caudal['data'].dt.date


Período em análise: 2026-07-22 até 2026-09-02
Registos de caudal: 9,193 (descartados 21,193 com caudal <= 0)
Dias que ficaram sem nenhuma leitura (10): 2026-07-27, 2026-07-28, 2026-07-31, 2026-08-01, 2026-08-02, 2026-08-03, 2026-08-04, 2026-08-05, 2026-08-17, 2026-08-19


In [3]:
# Verificar as primeiras linhas do CSV original
print("Primeiras 20 linhas do arquivo CSV:")
print(df_caudal.head(20))
print("\n" + "="*80)
print(f"Tipos de dados:\n{df_caudal.dtypes}")

Primeiras 20 linhas do arquivo CSV:
              datetime station  volume_acumulado_m3  caudal_l_s  altura_m  \
0  2026-07-22 10:40:00   ameal             5874.279        0.87     0.075   
1  2026-07-22 10:42:00   ameal             5874.398        0.99     0.076   
2  2026-07-22 10:44:00   ameal             5874.515        0.98     0.076   
3  2026-07-22 10:46:00   ameal             5874.640        1.04     0.077   
4  2026-07-22 10:48:00   ameal             5874.770        1.08     0.077   
5  2026-07-22 10:50:00   ameal             5874.911        1.18     0.077   
6  2026-07-22 10:52:00   ameal             5875.059        1.23     0.078   
7  2026-07-22 10:54:00   ameal             5875.210        1.26     0.078   
8  2026-07-22 10:56:00   ameal             5875.365        1.29     0.079   
9  2026-07-22 10:58:00   ameal             5875.522        1.31     0.079   
10 2026-07-22 11:00:00   ameal             5875.679        1.31     0.080   
11 2026-07-22 11:02:00   ameal          

In [4]:
import requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Carregar dados de precipitação
df_precip = pd.read_csv(PRECIP_CSV)
df_precip["precip_incremental"] = df_precip["precip_mm"]
df_precip['datetime'] = pd.to_datetime(df_precip['datetime'])
df_precip = df_precip[df_precip['datetime'] >= ANALISE_INICIO]
df_precip['dia'] = df_precip['datetime'].dt.date

# Agrupar precipitação por dia
precip_por_dia = df_precip.groupby('dia')['precip_incremental'].sum().reset_index()
precip_por_dia['dia'] = pd.to_datetime(precip_por_dia['dia'])

# Buscar dados Open-Meteo (coordenadas da estação escolhida)
START_DATE = precip_por_dia['dia'].min().strftime('%Y-%m-%d')
END_DATE = precip_por_dia['dia'].max().strftime('%Y-%m-%d')

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": "precipitation",
    "timezone": "Europe/Lisbon"
}

response = requests.get(url, params=params)
data = response.json()

df_openmeteo = pd.DataFrame({
    'datetime': pd.to_datetime(data['hourly']['time']),
    'precipitation': data['hourly']['precipitation'],
})

df_openmeteo['datetime'] = df_openmeteo['datetime'].dt.tz_localize(None)
df_openmeteo['dia'] = df_openmeteo['datetime'].dt.date

# Agrupar Open-Meteo por dia
openmeteo_por_dia = df_openmeteo.groupby('dia')['precipitation'].sum().reset_index()
openmeteo_por_dia['dia'] = pd.to_datetime(openmeteo_por_dia['dia'])

# Agrupar caudal por intervalos de 2 minutos
df_caudal_10min = (
    df_caudal.set_index('data').resample('2min')['caudal_m3_h'].mean()
    .dropna()
    .reset_index()
)
df_caudal_10min = df_caudal_10min[df_caudal_10min['caudal_m3_h'] > 0]

# Volume do dia: cada leitura dura 2 min, logo m³ = caudal (m³/h) × 2/60 h.
df_caudal_10min["volume_m3"] = df_caudal_10min["caudal_m3_h"] * (2 / 60)
caudal_por_dia = (
    df_caudal_10min.assign(dia=df_caudal_10min["data"].dt.normalize())
    .groupby("dia", as_index=False)["volume_m3"]
    .sum()
)

# Criar subplots com Plotly
fig = make_subplots(
    rows=3, cols=1,
    row_heights=[0.22, 0.46, 0.32],
    subplot_titles=('Precipitação (mm/dia)', 'Caudal (m³/h)', 'Caudal (m³/dia)'),
    vertical_spacing=0.08,
    specs=[[{"secondary_y": False}], [{"secondary_y": False}], [{"secondary_y": False}]],
    shared_xaxes=True
)

# Gráfico superior - Precipitação Sensor
fig.add_trace(
    go.Bar(
        x=precip_por_dia['dia'],
        y=precip_por_dia['precip_incremental'],
        name='Sensor',
        marker_color='darkviolet',
        opacity=0.4,
        hovertemplate='<b>Sensor</b><br>Data: %{x|%Y-%m-%d}<br>Precipitação: %{y:.2f} mm/dia<extra></extra>'
    ),
    row=1, col=1
)

# Gráfico superior - Precipitação Open-Meteo
fig.add_trace(
    go.Bar(
        x=openmeteo_por_dia['dia'],
        y=openmeteo_por_dia['precipitation'],
        name='Open-Meteo',
        marker_color='gold',
        opacity=0.4,
        hovertemplate='<b>Open-Meteo</b><br>Data: %{x-%m-%d}<br>Precipitação: %{y:.2f} mm/dia<extra></extra>'
    ),
    row=1, col=1
)

# Gráfico do meio - Caudal (m³/h)
fig.add_trace(
    go.Scatter(
        x=df_caudal_10min['data'],
        y=df_caudal_10min['caudal_m3_h'],
        name='Caudal',
        mode='lines',
        line=dict(color='navy', width=1),
        opacity=0.7,
        hovertemplate='<b>Caudal</b><br>Data/Hora: %{x|%Y-%m-%d %H:%M}<br>Caudal: %{y:.4f} m³/h<extra></extra>'
    ),
    row=2, col=1
)

# Gráfico de baixo - volume medido no dia (m³/dia)
fig.add_trace(
    go.Bar(
        x=caudal_por_dia["dia"],
        y=caudal_por_dia["volume_m3"],
        name="Caudal diário",
        marker_color="navy",
        opacity=0.75,
        hovertemplate="<b>Caudal diário</b><br>Data: %{x|%Y-%m-%d}<br>Volume: %{y:.1f} m³/dia<extra></extra>",
    ),
    row=3, col=1,
)

# Inverter eixo Y do gráfico superior (precipitação)
fig.update_yaxes(autorange="reversed", row=1, col=1)

# Atualizar layout
fig.update_layout(
    title_text=f'Precipitação e Caudal - {STATION_LABEL}',
    height=1100,
    showlegend=True,
    hovermode='closest',
    template='plotly_white'
)

fig.update_xaxes(title_text="Mês", row=3, col=1)
fig.update_yaxes(title_text="Precipitação (mm/dia)", row=1, col=1)
fig.update_yaxes(title_text="Caudal (m³/h)", row=2, col=1)
fig.update_yaxes(title_text="Caudal (m³/dia)", row=3, col=1)

fig.show()


In [5]:
# Exportar gráfico como HTML interativo
html_path = f"precipitacao_caudal_{ESTACAO}.html"
fig.write_html(html_path)
print(f"Gráfico exportado para {html_path}")
print("A pessoa só precisa abrir este arquivo no navegador e terá acesso a todos os recursos interativos!")

Gráfico exportado para precipitacao_caudal_ameal.html
A pessoa só precisa abrir este arquivo no navegador e terá acesso a todos os recursos interativos!


# CALCULO DAS AFLUENCIAS

# Afluências Indevidas (02:00-04:00)

In [6]:
# Filtrar dados das 02:00 às 04:00. Sem resample: a série já é de 2 min,
# e resample inventava NaN nos buracos (o Plotly desenhava-os em y=0).
df_caudal_02_04 = df_caudal[
    (df_caudal['data'].dt.time >= pd.to_datetime("02:00:00").time())
    & (df_caudal['data'].dt.time < pd.to_datetime("04:00:00").time())
    & (df_caudal['caudal_m3_h'] > 0)
].copy()
df_caudal_02_04_2min = df_caudal_02_04[['data', 'caudal_m3_h']].copy()

# Exibir informações
print(f"Dados filtrados (02:00-04:00): {len(df_caudal_02_04)} registros")
print(f"Após agrupamento de 2 min: {len(df_caudal_02_04_2min)} registros")
if len(df_caudal_02_04) > 0:
    print(f"Período: {df_caudal_02_04['data'].min()} até {df_caudal_02_04['data'].max()}")

Dados filtrados (02:00-04:00): 522 registros
Após agrupamento de 2 min: 522 registros
Período: 2026-08-08 03:24:00 até 2026-09-02 03:58:00


In [7]:
# Criar gráfico para dados de 02:00-04:00
fig_02_04 = make_subplots(
    rows=2, cols=1,
    row_heights=[0.3, 0.7],
    subplot_titles=('Precipitação (mm/dia)', 'Caudal (m³/h) - 02:00 às 04:00'),
    vertical_spacing=0.1,
    specs=[[{"secondary_y": False}], [{"secondary_y": False}]]
)

# Gráfico superior - Precipitação Sensor
fig_02_04.add_trace(
    go.Bar(
        x=precip_por_dia['dia'],
        y=precip_por_dia['precip_incremental'],
        name='Sensor',
        marker_color='darkviolet',
        opacity=0.4,
        hovertemplate='<b>Sensor</b><br>Data: %{x|%Y-%m-%d}<br>Precipitação: %{y:.2f} mm/dia<extra></extra>'
    ),
    row=1, col=1
)

# Gráfico superior - Precipitação Open-Meteo
fig_02_04.add_trace(
    go.Bar(
        x=openmeteo_por_dia['dia'],
        y=openmeteo_por_dia['precipitation'],
        name='Open-Meteo',
        marker_color='gold',
        opacity=0.4,
        hovertemplate='<b>Open-Meteo</b><br>Data: %{x|%Y-%m-%d}<br>Precipitação: %{y:.2f} mm/dia<extra></extra>'
    ),
    row=1, col=1
)

# Gráfico inferior - Caudal (02:00-04:00)
_plot_02_04 = df_caudal_02_04_2min.dropna(subset=['caudal_m3_h'])
_plot_02_04 = _plot_02_04[_plot_02_04['caudal_m3_h'] > 0]
if len(_plot_02_04) > 0:
    fig_02_04.add_trace(
        go.Scatter(
            x=_plot_02_04['data'],
            y=_plot_02_04['caudal_m3_h'],
            name='Caudal (02:00-04:00)',
            mode='markers',
            line=dict(color='navy', width=1),
            opacity=0.7,
            hovertemplate='<b>Caudal</b><br>Data/Hora: %{x|%Y-%m-%d %H:%M}<br>Caudal: %{y:.4f} m³/h<extra></extra>'
        ),
        row=2, col=1
    )

# Inverter eixo Y do gráfico superior (precipitação)
fig_02_04.update_yaxes(autorange="reversed", row=1, col=1)

# Atualizar layout
fig_02_04.update_layout(
    title_text=f'Precipitação e Caudal (02:00-04:00) - {STATION_LABEL}',
    height=800,
    showlegend=True,
    hovermode='closest',
    template='plotly_white'
)

fig_02_04.update_xaxes(title_text="Mês", row=2, col=1)
fig_02_04.update_yaxes(title_text="Precipitação (mm/dia)", row=1, col=1)
fig_02_04.update_yaxes(title_text="Caudal (m³/h)", row=2, col=1)

fig_02_04.show()

In [8]:
precip_por_dia

,dia,precip_incremental
0,2026-07-01,0.0
1,2026-07-02,0.0
2,2026-07-03,0.0
3,2026-07-04,0.0
4,2026-07-05,0.0
...,...,...
60,2026-08-30,0.0
61,2026-08-31,0.0
62,2026-09-01,0.1
63,2026-09-02,0.0


In [9]:
# Criar máscara para dias sem chuva considerando janela de 24h antes e depois
# Um dia só entra se não choveu nele, no dia anterior e no dia seguinte

import datetime

# Criar sets com dias sem chuva
dias_sem_chuva_sensor = set(precip_por_dia[precip_por_dia["precip_incremental"] == 0]['dia'].dt.date)
dias_sem_chuva_openmeteo = set(openmeteo_por_dia[openmeteo_por_dia['precipitation'] == 0]['dia'].dt.date)

# Dias sem chuva em ambas as fontes
dias_sem_chuva_base = dias_sem_chuva_sensor.union(dias_sem_chuva_openmeteo)

# Filtrar para incluir apenas dias onde não choveu nos últimos X dias
DIAS_ANTES = 5   # Número de dias sem chuva ANTES necessários
dias_sem_chuva_janela = set()

for dia in dias_sem_chuva_base:
    # Verificar se TODOS os dias anteriores (de 1 até DIAS_ANTES) também não tiveram chuva
    todos_dias_anteriores_sem_chuva = True
    
    for i in range(1, DIAS_ANTES + 1):
        dia_anterior = dia - datetime.timedelta(days=i)
        if dia_anterior not in dias_sem_chuva_base:
            todos_dias_anteriores_sem_chuva = False
            break
    
    if todos_dias_anteriores_sem_chuva:
        dias_sem_chuva_janela.add(dia)

# Filtrar caudal 02:00-04:00 apenas para dias sem chuva (com janela)
df_caudal_sem_chuva = df_caudal_02_04[df_caudal_02_04['dia'].isin(dias_sem_chuva_janela)].copy()

# Agrupar por dia e pegar o valor MÍNIMO de caudal em cada madrugada.
# Leituras a zero ja foram retiradas no carregamento.
df_caudal_sem_chuva = df_caudal_sem_chuva[df_caudal_sem_chuva['caudal_m3_h'] > 0]
if len(df_caudal_sem_chuva) > 0:
    idx_min = df_caudal_sem_chuva.groupby('dia')['caudal_m3_h'].idxmin()
    df_caudal_sem_chuva_2min = df_caudal_sem_chuva.loc[
        idx_min, ['dia', 'data', 'caudal_m3_h']
    ].reset_index(drop=True)
else:
    df_caudal_sem_chuva_2min = df_caudal_sem_chuva.iloc[0:0][['dia', 'data', 'caudal_m3_h']]

# IMPORTANTE: Filtrar por caudal máximo DEPOIS do agrupamento
CAUDAL_MAXIMO = 99999 # esse filtro aqui é porque teve uma chuva muito forte nos dados, e mesmo sem chuva o cauldal se manteve alto por muito tempo mas era ruido da chuva
df_caudal_sem_chuva_2min_filtrado = df_caudal_sem_chuva_2min[
    (df_caudal_sem_chuva_2min["caudal_m3_h"] > 0)
    & (df_caudal_sem_chuva_2min["caudal_m3_h"] < CAUDAL_MAXIMO)
].copy()
print(f"Total de dias sem chuva (sem janela): {len(dias_sem_chuva_base)}")
print(f"Total de dias sem chuva (com janela de {DIAS_ANTES} dias antes): {len(dias_sem_chuva_janela)}")
print(f"Registros de caudal 02:00-04:00 em dias sem chuva: {len(df_caudal_sem_chuva)}")
print(f"Após agrupamento por dia (mínimo): {len(df_caudal_sem_chuva_2min)} registros")
print(f"Após filtro (< {CAUDAL_MAXIMO} m³/h): {len(df_caudal_sem_chuva_2min_filtrado)} registros")

if len(df_caudal_sem_chuva_2min_filtrado) > 0:
    caudal_medio = df_caudal_sem_chuva_2min_filtrado['caudal_m3_h'].mean()
    caudal_minimo = df_caudal_sem_chuva_2min_filtrado['caudal_m3_h'].min()
    caudal_maximo = df_caudal_sem_chuva_2min_filtrado['caudal_m3_h'].max()
    print(f"Caudal médio (dos mínimos diários): {caudal_medio:.6f} m³/h")
    print(f"Caudal mínimo absoluto: {caudal_minimo:.6f} m³/h")
    print(f"Caudal máximo (dos mínimos): {caudal_maximo:.6f} m³/h")

# Criar gráfico de Afluências Indiretas (sem chuva, filtrado)
fig_afluencias_indiretas = make_subplots(
    rows=2, cols=1,
    row_heights=[0.3, 0.7],
    subplot_titles=('Precipitação (mm/dia)', f'Caudal (m³/h) - Afluências Indiretas 02:00-04:00 vs Caudal medido'),
    vertical_spacing=0.1,
    specs=[[{"secondary_y": False}], [{"secondary_y": False}]]
)

# Gráfico superior - Precipitação Sensor
fig_afluencias_indiretas.add_trace(
    go.Bar(
        x=precip_por_dia['dia'],
        y=precip_por_dia['precip_incremental'],
        name='Sensor',
        marker_color='darkviolet',
        opacity=0.4,
        hovertemplate='<b>Sensor</b><br>Data: %{x|%Y-%m-%d}<br>Precipitação: %{y:.2f} mm/dia<extra></extra>'
    ),
    row=1, col=1
)

# Gráfico superior - Precipitação Open-Meteo
fig_afluencias_indiretas.add_trace(
    go.Bar(
        x=openmeteo_por_dia['dia'],
        y=openmeteo_por_dia['precipitation'],
        name='Open-Meteo',
        marker_color='gold',
        opacity=0.4,
        hovertemplate='<b>Open-Meteo</b><br>Data: %{x|%Y-%m-%d}<br>Precipitação: %{y:.2f} mm/dia<extra></extra>'
    ),
    row=1, col=1
)

# Gráfico inferior - Caudal medido (m³/h), serie completa de 2 min, para
# comparar os minimos das madrugadas com o caudal que o sensor registou.
fig_afluencias_indiretas.add_trace(
    go.Scatter(
        x=df_caudal_10min['data'],
        y=df_caudal_10min['caudal_m3_h'],
        name='Caudal medido',
        mode='lines',
        line=dict(color='navy', width=1),
        opacity=0.45,
        hovertemplate='<b>Caudal medido</b><br>Data/Hora: %{x|%Y-%m-%d %H:%M}<br>Caudal: %{y:.4f} m³/h<extra></extra>'
    ),
    row=2, col=1
)

# Gráfico inferior - Caudal filtrado (apenas dias sem chuva, < CAUDAL_MAXIMO)
if len(df_caudal_sem_chuva_2min_filtrado) > 0:
    fig_afluencias_indiretas.add_trace(
        go.Scatter(
            x=df_caudal_sem_chuva_2min_filtrado['data'],
            y=df_caudal_sem_chuva_2min_filtrado['caudal_m3_h'],
            name=f'Caudal < {CAUDAL_MAXIMO} m³/h',
            mode='markers',
            marker=dict(color='green', size=9, line=dict(color='darkgreen', width=1)),
            opacity=0.7,
            hovertemplate='<b>Caudal</b><br>Data/Hora: %{x|%Y-%m-%d %H:%M}<br>Caudal: %{y:.6f} m³/h<extra></extra>'
        ),
        row=2, col=1
    )
    
    # Adicionar linha horizontal com o valor médio
    MEDIA_AFLUENCIAS_INDIRETAS = df_caudal_sem_chuva_2min_filtrado['caudal_m3_h'].mean()
    AFLUENCIA_INDIRETA_MINIMO_DOS_MINIMOS_M3H = df_caudal_sem_chuva_2min_filtrado['caudal_m3_h'].min()
        
        # Adicionar linha horizontal com o valor mínimo
    fig_afluencias_indiretas.add_hline(
        y=AFLUENCIA_INDIRETA_MINIMO_DOS_MINIMOS_M3H,
        line_dash="dot",
        line_color="green",
        annotation_text=f"Mínimo: {AFLUENCIA_INDIRETA_MINIMO_DOS_MINIMOS_M3H:.6f} m³/h",
        annotation_position="right",
        row=2, col=1
    )
    fig_afluencias_indiretas.add_hline(
        y=MEDIA_AFLUENCIAS_INDIRETAS,
        line_dash="dash",
        line_color="red",
        annotation_text=f"Média: {MEDIA_AFLUENCIAS_INDIRETAS:.6f} m³/h",
        annotation_position="right",
        row=2, col=1
    )

    # Folga no eixo; o maximo considera tambem o caudal medido para a
    # linha nao ficar cortada nos picos de chuva.
    y_max = max(
        df_caudal_sem_chuva_2min_filtrado['caudal_m3_h'].max(),
        df_caudal_10min['caudal_m3_h'].max(),
    )
    fig_afluencias_indiretas.update_yaxes(
        range=[0, 1.1 * y_max], row=2, col=1
    )

# Inverter eixo Y do gráfico superior (precipitação)
fig_afluencias_indiretas.update_yaxes(autorange="reversed", row=1, col=1)

# Atualizar layout
fig_afluencias_indiretas.update_layout(
    title_text=f'Precipitação e Afluências indiretas 02:00-04:00 (sem chuva) - {STATION_LABEL}',
    height=800,
    showlegend=True,
    hovermode='closest',
    template='plotly_white'
)

fig_afluencias_indiretas.update_xaxes(title_text="Mês", row=2, col=1)
fig_afluencias_indiretas.update_yaxes(title_text="Precipitação (mm/dia)", row=1, col=1)
fig_afluencias_indiretas.update_yaxes(title_text="Caudal (m³/h)", row=2, col=1)

fig_afluencias_indiretas.show()

Total de dias sem chuva (sem janela): 46
Total de dias sem chuva (com janela de 5 dias antes): 16
Registros de caudal 02:00-04:00 em dias sem chuva: 41
Após agrupamento por dia (mínimo): 2 registros
Após filtro (< 99999 m³/h): 2 registros
Caudal médio (dos mínimos diários): 1.836000 m³/h
Caudal mínimo absoluto: 1.656000 m³/h
Caudal máximo (dos mínimos): 2.016000 m³/h


# Afluencias indevidas diretas

In [10]:
MEDIA_AFLUENCIAS_INDIRETAS
AFLUENCIA_INDIRETA_MINIMO_DOS_MINIMOS_M3H


np.float64(1.656)

In [11]:
# Filtrar df_caudal completo (todos os horários) apenas para dias sem chuva
df_caudal_diretas = df_caudal[df_caudal['dia'].isin(dias_sem_chuva_janela)].copy()

df_caudal_diretas['caudal_esgoto_m3_h'] = df_caudal_diretas['caudal_m3_h'] - AFLUENCIA_INDIRETA_MINIMO_DOS_MINIMOS_M3H
# Agrupar por intervalos de 2 minutos
df_caudal_diretas_2min = (
    df_caudal_diretas.set_index('data').resample('2min')['caudal_esgoto_m3_h'].mean()
    .dropna()
    .reset_index()
)
print(f"Média de afluências indiretas subtraída: {AFLUENCIA_INDIRETA_MINIMO_DOS_MINIMOS_M3H:.6f} m³/h")
print(f"Registros de caudal (todos horários) em dias sem chuva: {len(df_caudal_diretas)}")
print(f"Após agrupamento de 2 min: {len(df_caudal_diretas_2min)} registros")

if len(df_caudal_diretas_2min) > 0:
    print(f"Caudal direto médio: {df_caudal_diretas_2min['caudal_esgoto_m3_h'].mean():.6f} m³/h")
    print(f"Caudal direto mínimo: {df_caudal_diretas_2min['caudal_esgoto_m3_h'].min():.6f} m³/h")
    print(f"Caudal direto máximo: {df_caudal_diretas_2min['caudal_esgoto_m3_h'].max():.6f} m³/h")

Média de afluências indiretas subtraída: 1.656000 m³/h
Registros de caudal (todos horários) em dias sem chuva: 778
Após agrupamento de 2 min: 778 registros
Caudal direto médio: 1.915172 m³/h
Caudal direto mínimo: -0.288000 m³/h
Caudal direto máximo: 6.840000 m³/h


In [12]:
# Criar gráfico de Caudal de Esgoto (caudal - afluências indiretas, sem chuva)
fig_caudal_esgoto = make_subplots(
    rows=2, cols=1,
    row_heights=[0.3, 0.7],
    subplot_titles=('Precipitação (mm/dia)', 'Caudal de Esgoto (m³/h) = Caudal Total - Afluências Indiretas - CHUVA ZERO'),
    vertical_spacing=0.1,
    specs=[[{"secondary_y": False}], [{"secondary_y": False}]],
    shared_xaxes=True
)

# Gráfico superior - Precipitação Sensor
fig_caudal_esgoto.add_trace(
    go.Bar(
        x=precip_por_dia['dia'],
        y=precip_por_dia['precip_incremental'],
        name='Sensor',
        marker_color='darkviolet',
        opacity=0.4,
        hovertemplate='<b>Sensor</b><br>Data: %{x|%Y-%m-%d}<br>Precipitação: %{y:.2f} mm/dia<extra></extra>'
    ),
    row=1, col=1
)

# Gráfico superior - Precipitação Open-Meteo
fig_caudal_esgoto.add_trace(
    go.Bar(
        x=openmeteo_por_dia['dia'],
        y=openmeteo_por_dia['precipitation'],
        name='Open-Meteo',
        marker_color='gold',
        opacity=0.4,
        hovertemplate='<b>Open-Meteo</b><br>Data: %{x|%Y-%m-%d}<br>Precipitação: %{y:.2f} mm/dia<extra></extra>'
    ),
    row=1, col=1
)

# Gráfico inferior - Caudal de Esgoto (dias sem chuva, todos os horários)
if len(df_caudal_diretas_2min) > 0:
    fig_caudal_esgoto.add_trace(
        go.Scatter(
            x=df_caudal_diretas_2min['data'],
            y=df_caudal_diretas_2min['caudal_esgoto_m3_h'],
            name='Caudal de Esgoto',
            mode='lines',
            line=dict(color='blue', width=1),
            opacity=0.7,
            hovertemplate='<b>Caudal de Esgoto</b><br>Data/Hora: %{x|%Y-%m-%d %H:%M}<br>Caudal: %{y:.6f} m³/h<extra></extra>'
        ),
        row=2, col=1
    )
    
    # Adicionar linha horizontal com o valor médio
    caudal_esgoto_medio = df_caudal_diretas_2min['caudal_esgoto_m3_h'].mean()
    fig_caudal_esgoto.add_hline(
        y=caudal_esgoto_medio,
        line_dash="dash",
        line_color="red",
        annotation_text=f"Média: {caudal_esgoto_medio:.6f} m³/h",
        annotation_position="right",
        row=2, col=1
    )
    
    # Adicionar linha em y=0 para referência
    fig_caudal_esgoto.add_hline(
        y=0,
        line_dash="dot",
        line_color="gray",
        annotation_text="Zero",
        annotation_position="left",
        row=2, col=1
    )

# Inverter eixo Y do gráfico superior (precipitação)
fig_caudal_esgoto.update_yaxes(autorange="reversed", row=1, col=1)

# Atualizar layout
fig_caudal_esgoto.update_layout(
    title_text=f'Precipitação e Caudal de Esgoto (Caudal - {AFLUENCIA_INDIRETA_MINIMO_DOS_MINIMOS_M3H:.6f} m³/h, sem chuva) - {STATION_LABEL}',
    height=800,
    showlegend=True,
    hovermode='closest',
    template='plotly_white'
)

fig_caudal_esgoto.update_xaxes(title_text="Mês", row=2, col=1)
fig_caudal_esgoto.update_yaxes(title_text="Precipitação (mm/dia)", row=1, col=1)
fig_caudal_esgoto.update_yaxes(title_text="Caudal de Esgoto (m³/s)", row=2, col=1)

fig_caudal_esgoto.show()


In [13]:
# Calcular caudal de afluências diretas (apenas dias COM chuva)
# Afluência Direta = Caudal Total - Caudal Esgoto - Afluências Indiretas

# Primeiro, criar conjunto de dias COM chuva
dias_com_chuva_sensor = set(precip_por_dia[precip_por_dia["precip_incremental"] > 0]['dia'].dt.date)
dias_com_chuva_openmeteo = set(openmeteo_por_dia[openmeteo_por_dia['precipitation'] > 0]['dia'].dt.date)

# Dias com chuva em pelo menos uma das fontes
dias_com_chuva = dias_com_chuva_sensor.union(dias_com_chuva_openmeteo)

# Filtrar caudal total apenas para dias com chuva
df_afluencias_diretas = df_caudal[df_caudal['dia'].isin(dias_com_chuva)].copy()

# Calcular caudal médio de esgoto (da análise anterior)
CAUDAL_MEDIO_ESGOTO = df_caudal_diretas_2min['caudal_esgoto_m3_h'].mean()

# Calcular afluência direta
# Afluência Direta = Caudal Total - Caudal Esgoto Médio - Afluências Indiretas Médias
df_afluencias_diretas['caudal_afluencia_direta_m3_h'] = (
    df_afluencias_diretas['caudal_m3_h'] - CAUDAL_MEDIO_ESGOTO - AFLUENCIA_INDIRETA_MINIMO_DOS_MINIMOS_M3H
)

# Filtrar valores negativos (quando caudal total é menor que a soma das afluências)
df_afluencias_diretas = df_afluencias_diretas[df_afluencias_diretas['caudal_afluencia_direta_m3_h'] >= 0].copy()

# Agrupar por intervalos de 2 minutos
df_afluencias_diretas_2min = (
    df_afluencias_diretas.set_index('data').resample('2min')['caudal_afluencia_direta_m3_h'].mean()
    .dropna()
    .reset_index()
)

print(f"Caudal médio de esgoto: {CAUDAL_MEDIO_ESGOTO:.6f} m³/h")
print(f"Média de afluências indiretas: {AFLUENCIA_INDIRETA_MINIMO_DOS_MINIMOS_M3H:.6f} m³/h")
print(f"Total de dias com chuva: {len(dias_com_chuva)}")
print(f"Registros de caudal em dias com chuva: {len(df_afluencias_diretas)}")
print(f"Após agrupamento de 2 min: {len(df_afluencias_diretas_2min)} registros")

if len(df_afluencias_diretas_2min) > 0:
    caudal_direto_medio = df_afluencias_diretas_2min['caudal_afluencia_direta_m3_h'].mean()
    print(f"Caudal de afluências diretas médio: {caudal_direto_medio:.6f} m³/h")
    print(f"Caudal de afluências diretas mínimo: {df_afluencias_diretas_2min['caudal_afluencia_direta_m3_h'].min():.6f} m³/h")
    print(f"Caudal de afluências diretas máximo: {df_afluencias_diretas_2min['caudal_afluencia_direta_m3_h'].max():.6f} m³/h")

# Criar gráfico de Afluências Diretas
fig_afluencias_diretas = make_subplots(
    rows=2, cols=1,
    row_heights=[0.3, 0.7],
    subplot_titles=('Precipitação (mm/dia)', 'Afluências Diretas, Afluências Indiretas e Caudal medido (m³/h)'),
    vertical_spacing=0.1,
    specs=[[{"secondary_y": False}], [{"secondary_y": False}]]
)

# Gráfico superior - Precipitação Sensor
fig_afluencias_diretas.add_trace(
    go.Bar(
        x=precip_por_dia['dia'],
        y=precip_por_dia['precip_incremental'],
        name='Sensor',
        marker_color='darkviolet',
        opacity=0.4,
        hovertemplate='<b>Sensor</b><br>Data: %{x|%Y-%m-%d}<br>Precipitação: %{y:.2f} mm/dia<extra></extra>'
    ),
    row=1, col=1
)

# Gráfico superior - Precipitação Open-Meteo
fig_afluencias_diretas.add_trace(
    go.Bar(
        x=openmeteo_por_dia['dia'],
        y=openmeteo_por_dia['precipitation'],
        name='Open-Meteo',
        marker_color='gold',
        opacity=0.4,
        hovertemplate='<b>Open-Meteo</b><br>Data: %{x|%Y-%m-%d}<br>Precipitação: %{y:.2f} mm/dia<extra></extra>'
    ),
    row=1, col=1
)

# Gráfico inferior - Caudal medido (m³/h), serie completa de 2 min, para
# comparar a afluencia direta com o caudal que o sensor registou na mesma hora.
fig_afluencias_diretas.add_trace(
    go.Scatter(
        x=df_caudal_10min['data'],
        y=df_caudal_10min['caudal_m3_h'],
        name='Caudal medido',
        mode='lines',
        line=dict(color='navy', width=1),
        opacity=0.45,
        hovertemplate='<b>Caudal medido</b><br>Data/Hora: %{x|%Y-%m-%d %H:%M}<br>Caudal: %{y:.4f} m³/h<extra></extra>'
    ),
    row=2, col=1
)

# Gráfico inferior - Afluências Indiretas (minimos 02:00-04:00 em dias sem
# chuva), no mesmo eixo X das diretas para comparar os dois periodos.
if len(df_caudal_sem_chuva_2min_filtrado) > 0:
    fig_afluencias_diretas.add_trace(
        go.Scatter(
            x=df_caudal_sem_chuva_2min_filtrado['data'],
            y=df_caudal_sem_chuva_2min_filtrado['caudal_m3_h'],
            name='Afluências Indiretas',
            mode='markers',
            marker=dict(color='green', size=8, line=dict(color='darkgreen', width=1)),
            opacity=0.8,
            hovertemplate='<b>Afluências Indiretas</b><br>Data/Hora: %{x|%Y-%m-%d %H:%M}<br>Caudal: %{y:.6f} m³/h<extra></extra>'
        ),
        row=2, col=1
    )

# Gráfico inferior - Caudal de Afluências Diretas
if len(df_afluencias_diretas_2min) > 0:
    fig_afluencias_diretas.add_trace(
        go.Scatter(
            x=df_afluencias_diretas_2min['data'],
            y=df_afluencias_diretas_2min['caudal_afluencia_direta_m3_h'],
            name='Afluências Diretas',
            mode='lines',
            line=dict(color='red', width=1),
            opacity=0.7,
            hovertemplate='<b>Afluências Diretas</b><br>Data/Hora: %{x|%Y-%m-%d %H:%M}<br>Caudal: %{y:.6f} m³/h<extra></extra>'
        ),
        row=2, col=1
    )
    
    # Adicionar linha horizontal com o valor médio
    caudal_direto_medio = df_afluencias_diretas_2min['caudal_afluencia_direta_m3_h'].mean()
    fig_afluencias_diretas.add_hline(
        y=caudal_direto_medio,
        line_dash="dash",
        line_color="orange",
        annotation_text=f"Média: {caudal_direto_medio:.6f} m³/h",
        annotation_position="right",
        row=2, col=1
    )

# Inverter eixo Y do gráfico superior (precipitação)
fig_afluencias_diretas.update_yaxes(autorange="reversed", row=1, col=1)

# Atualizar layout
fig_afluencias_diretas.update_layout(
    title_text=f'Precipitação, Afluências Diretas (Caudal Total - Esgoto - Indiretas), Indiretas e Caudal medido - {STATION_LABEL}',
    height=800,
    showlegend=True,
    hovermode='closest',
    template='plotly_white',
    xaxis=dict(matches='x2')
)

fig_afluencias_diretas.update_xaxes(title_text="Mês", row=2, col=1)
fig_afluencias_diretas.update_yaxes(title_text="Precipitação (mm/dia)", row=1, col=1)
fig_afluencias_diretas.update_yaxes(title_text="Caudal (m³/h)", row=2, col=1)

fig_afluencias_diretas.show()

Caudal médio de esgoto: 1.915172 m³/h
Média de afluências indiretas: 1.656000 m³/h
Total de dias com chuva: 19
Registros de caudal em dias com chuva: 3654
Após agrupamento de 2 min: 3654 registros
Caudal de afluências diretas médio: 2.981675 m³/h
Caudal de afluências diretas mínimo: 0.028828 m³/h
Caudal de afluências diretas máximo: 66.916828 m³/h
